In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
# from any_llm import completion, responses, list_models
from panel import llm, paths
from panel.analysis import plots, validate
from panel.describe.template import read_table
from panel.run import RunSpec, ResultStore
from panel.topic import load_topic, make_response_models

load_dotenv()

## Experimen Setup

In [ ]:
# Run setup - the only place a run is declared

TOPIC = load_topic("election_2021", paths.TOPICS_DIR)
VProb, VPick = make_response_models(TOPIC)
DESCRIPTION = "self_description_v1"

# The descriptions were rendered at cleanup (datasets/sod/code/describe.py) and are
# joined here, so the prompt a run sends is a stored artifact rather than whatever the
# describer happened to be today. read_table refuses a table that does not cover the data.
people = pd.read_csv(paths.DATA_DIR / "clean_data.csv", index_col=0)
data = people.join(read_table(DESCRIPTION, people.index))

BASE = RunSpec(
    name="sod",
    topic=TOPIC.name,
    data=data,
    description=DESCRIPTION,
    question=TOPIC.question,
    instructions=TOPIC.instructions,
    client_type=llm.OpenRouterClient,
    model="gpt-4.1-nano",
    response_model=VProb,
    temperature=0,
)

SPECS = {
    "nano":        BASE,
    "4o-mini":     BASE.variant(model="gpt-4o-mini"),
    "4.1":         BASE.variant(model="gpt-4.1"),
    "4o":          BASE.variant(model="gpt-4o"),
    "gemini":      BASE.variant(model="gemini-3-flash-preview"),
    "sonnet":      BASE.variant(model="claude-sonnet-4-5"),
    "4o-openai":   BASE.variant(client_type=llm.OpenAIClient, model="gpt-4o"),
    "4o-pick":     BASE.variant(client_type=llm.OpenAIClient, model="gpt-4o", response_model=VPick),
    "nano-trial":  BASE.variant(sample=200),
}

SPEC = SPECS["nano"]

In [ ]:
STORE = ResultStore(paths.RESULTS_DIR, SPEC.topic)
CLIENT = SPEC.make_client()
respondents = SPEC.rows()

print(f"{SPEC.model} via {type(CLIENT).__name__}, t={SPEC.temperature}, {len(respondents)} of {len(SPEC.data)} rows")
print(f"-> {STORE.paths(SPEC.stem).results_csv}")

In [ ]:
# Prompts

# instructions = """
# Jsi expertní AI asistent specializovaný na analýzu českého politického chování. Tvým úkolem je na základě demografického a postojového profilu respondenta odhadnout jeho volební chování ve volbách do Poslanecké sněmovny Parlamentu ČR v roce 2021.
#
# Tvůj výstup MUSÍ být JSON objekt, který přesně odpovídá Pydantic modelu `VotingResult`. Jiný formát není přípustný.
#
# Dodržuj tato pravidla:
# 1.  Analyzuj VŠECHNY poskytnuté informace o respondentovi (věk, vzdělání, bydliště, příjem, postoje k EU/NATO atd.).
# 2.  Na základě analýzy odhadni dvě klíčové věci:
#     a) Jaká je pravděpodobnost, že respondent vůbec šel k volbám.
#     b) Pokud volil, jaké jsou pravděpodobnosti pro jednotlivé politické strany.
# 3.  Vygeneruj JSON, který bude validní oproti poskytnutým Pydantic modelům.
# 4.  Dbej na to, aby součet pravděpodobností v objektu `voted_or_not` byl přesně 1.0.
# 5.  Dbej na to, aby součet pravděpodobností VŠECH stran v seznamu `parties` byl přesně 1.0."""

## Prompt, Model and Response Showcase

In [ ]:
respondent = data.iloc[143]
prompt = SPEC.prompt(respondent)
prompt

In [ ]:
voted = CLIENT.call_llm(SPEC.model, prompt, SPEC.response_model, SPEC.temperature, SPEC.instructions)
voted

In [ ]:
# Pure GPT call
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.responses.parse(
    model=SPEC.model,
    temperature=SPEC.temperature,
    instructions=SPEC.instructions,
    input=prompt,
    text_format=SPEC.response_model
)

voted = response.output_parsed
response.output_parsed

In [ ]:
# PROVIDER = "ollama"
# MODEL = "jean-luc/tiger-gemma-9b-v3:fp16"

In [ ]:
# # any llm respons - Structure does not work
# response = responses(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     #instructions=INSTRUCTIONS,
#     input_data=prompt,
#     text=VOTING_RESULT
# )

In [ ]:
# # any llm completion - Only JSON
# response = completion(
#     provider=PROVIDER,
#     model=MODEL,
#     temperature=TEMPERATURE,
#     messages=[
#         # {"role": "system", "content": INSTRUCTIONS},
#         {"role": "user", "content": prompt }
#     ],
#     response_format=VOTING_RESULT
# )
# voted = VotingProbabilities.model_validate_json(response.choices[0].message.content)
# response.choices[0].message

In [ ]:
p = 0
print(voted.voted_or_not)
for party_vote in voted.parties:
    print(f"{party_vote.name}: {party_vote.probability}")
    p += party_vote.probability
print(f"p_sum = {p}")

## Election Simulation Experiment

In [ ]:
voting_results, skipped = llm.run_llm_batch(
    data=respondents,
    prompt_creator=SPEC.prompt,
    response_model=SPEC.response_model,
    model=SPEC.model, temperature=SPEC.temperature,
    client=CLIENT, instructions=SPEC.instructions
)

print(f"\nProcessed: {len(voting_results)}")
print(f"Failed/Skipped: {len(skipped)}")

In [ ]:
## Rerun skipped results
MAX_RERUN_ROUNDS = 5
rounds = 0
skipped_before = -1

while skipped and len(skipped) != skipped_before and rounds < MAX_RERUN_ROUNDS:
    skipped_before = len(skipped)
    rounds += 1

    skipped_voting_results, skipped = llm.run_llm_batch(
        data=respondents.loc[skipped],
        prompt_creator=SPEC.prompt,
        response_model=SPEC.response_model,
        model=SPEC.model, temperature=SPEC.temperature,
        client=CLIENT, instructions=SPEC.instructions
    )
    voting_results.update(skipped_voting_results)
    print(f"Rerun {rounds}: {len(skipped)} still skipped")

if skipped:
    raise RuntimeError(f"{len(skipped)} respondents unprocessed after {rounds} rerun rounds: {skipped}")

## Save Results

In [ ]:
NORMALIZE = TOPIC.normalize_groups

STORE.save(SPEC, voting_results, normalize=NORMALIZE)
paths = STORE.save(SPEC, voting_results, normalize=NORMALIZE, timestamped=False)
paths

## Result Evaluation

Validate the quality of the voting results by checking for common issues.

In [14]:
# Evaluate voting results quality
tol = 0.01
bad_voted_sum, bad_party_probs_sum, duplicate_parties = validate.evaluate_voting_results(voting_results, tol)

# Print details
if bad_voted_sum:
    print(f"[WARN] {len(bad_voted_sum)} respondents where voted+not_voted != 1 (|delta|>{tol}):")
    for rid, v, nv, s in bad_voted_sum[:20]:
        print(f" - ID={rid}: voted={v:.3f}, not_voted={nv:.3f}, sum={s:.3f}")
    if len(bad_voted_sum) > 20:
        print(f" ... and {len(bad_voted_sum) - 20} more")
else:
    print("[OK] All respondents have voted+not_voted summing to 1 within tolerance.")

if bad_party_probs_sum:
    print(f"[WARN] {len(bad_party_probs_sum)} respondents where party probabilities don't sum to 1 (|delta|>{tol}):")
    for rid, s, v in bad_party_probs_sum[:20]:
        print(f" - ID={rid}: voted={v}, prob_sum={s:.3f}")
    if len(bad_party_probs_sum) > 20:
        print(f" ... and {len(bad_party_probs_sum) - 20} more")
else:
    print("[OK] All respondents have party probabilities summing to 1 within tolerance (when parties present).")

if duplicate_parties:
    print(f"[WARN] {len(duplicate_parties)} respondents with duplicate party names:")
    for rid, dups in duplicate_parties[:20]:
        print(f" - ID={rid}: duplicates={dups}")
    if len(duplicate_parties) > 20:
        print(f" ... and {len(duplicate_parties) - 20} more")
else:
    print("[OK] No duplicate party names found in any respondent's parties list.")
# Visualize party probability errors
if bad_party_probs_sum:
    plots.visualize_party_errors([abs(1 - s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")

[OK] All respondents have voted+not_voted summing to 1 within tolerance.
[OK] All respondents have party probabilities summing to 1 within tolerance (when parties present).
[WARN] 5 respondents with duplicate party names:
 - ID=3530: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=1586: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=2227: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=3310: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
 - ID=1775: duplicates=['Koalice PIRÁTI a STAROSTOVÉ']
No party probability errors to visualize.


In [15]:
# Visualize party probability errors
if bad_party_probs_sum:
    plots.visualize_party_errors([abs(1-s) for _, s, _ in bad_party_probs_sum])
else:
    print("No party probability errors to visualize.")

No party probability errors to visualize.
